In [2]:
##Evaluation 1 
#Hier werden Antworten mit dem LLaMA Basis Modell und dem gefeintuneten Adapter generiert.
#Als Referenz zum Generieren dient ein Evaluations-Datensatz mit 100 Nachrichtenpaaren.

In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model_path = r"c:\401\models\llama-3.2-3b-hf"
adapter_path = r"out_pilot_###3b/checkpoint-2000"  

eval_path = "eval_set_100.csv"
output_path = "eval_generations_100.csv"

assert torch.cuda.is_available() #Cuda test
print(torch.cuda.get_device_name(0))

In [ ]:
#Datensatz laden
eval_df = pd.read_csv(eval_path)

print(eval_df.columns)
print(len(eval_df))

#Input und Reference test
eval_df["input"] = eval_df["input"].astype(str)
eval_df["reference"] = eval_df["reference"].astype(str)

eval_df.head()

In [3]:
#Prompt in User-Assistant formatieren
def make_prompt(user_input: str) -> str:
    return f"User: {user_input.strip()}\nAssistant:"

In [4]:
#Generieren, mit 80 Tokens limit

def generate_only_new(model, tokenizer, prompt, max_new_tokens=80):
    model.eval()
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    cut = inputs["input_ids"].shape[-1]
    gen_ids = out[0][cut:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [5]:
#Tokenizer laden
tok = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

In [6]:
#Base Modell laden
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    dtype=torch.float16,
    local_files_only=True
)

base_answers = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Generating BASE"):
    prompt = make_prompt(row["input"])
    answer = generate_only_new(base_model, tok, prompt)
    base_answers.append(answer)

eval_df["base_answer"] = base_answers

del base_model
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Generating BASE: 100%|███████████████████████████████████████████████████████████████| 100/100 [07:10<00:00,  4.30s/it]


In [7]:
#FT Modell laden

offload_dir = r"c:\hf-offload"
os.makedirs(offload_dir, exist_ok=True)

base_cpu = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map={"": "cpu"},
    dtype=torch.float16,
    local_files_only=True
)

ft = PeftModel.from_pretrained(
    base_cpu,
    adapter_path,
    offload_dir=offload_dir,
)

ft_merged = ft.merge_and_unload().eval().to("cuda")

ft_answers = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Generating FT"):
    prompt = make_prompt(row["input"])
    answer = generate_only_new(ft_merged, tok, prompt)
    ft_answers.append(answer)

eval_df["ft_answer"] = ft_answers

del base_cpu, ft, ft_merged
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Generating FT: 100%|█████████████████████████████████████████████████████████████████| 100/100 [02:28<00:00,  1.49s/it]


In [ ]:
#Datensatz speichern

eval_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Gespeichert:", output_path)
eval_df[["input", "reference", "base_answer", "ft_answer"]].head()

In [ ]:
eval_df